# `02_conditional_edge.ipynb`
1. **state의 변화를 주지 않음** (node가 아님)
2. 현재 상황의 state 혹은 다른 기준을 바탕으로, 특정 값을 `return` 함
3. 해당 값과 매칭된 node가 실행됨

## 예시
- State
    - `message: str`
    - `is_ok: bool`
    - `next_step: str`

- Node
    - `process_message` : 현재 메세지에 `error`라는 단어가 있는지 체크
        - `error` 가 있으면, `is_ok`를 `False`. 없으면 `True`
        - `message` 에 `'admin'` 이라는 영단어가 들어있다면, `is_ok` 를 `'admin'` 으로 세팅
    - A: 에러 없으면, `normal_node`
    - B: 에러 있으면, `error_node`
    - C: `is_ok` 가 `'admin'` 이면, `call_admin_node`
    - `call_admin` 노드는 `next_step` 을 `관리자 호출!!` 로 세팅후 종료


- Conditional Edge
    - `is_ok` 의 값에 따라 다음 노드를 결정


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# State 정의
class SystemState(TypedDict):
    message: str
    is_ok: bool
    next_step: str

# {'message': '좋아좋아', 'is_ok': True, 'next_step': '계속 진행하세요'}

In [ ]:
# Node 정의
def process_message(state: SystemState):
    print('---메세지 처리 노드---', state)
    msg = state['message']
    if ('error' in msg.lower()) or ('에러' in msg):
        result = False
    elif 'admin' in msg.lower():
        result = 'admin'
    else:
        result = True
        
    # Node 는 전체 state를 리턴하지 않음. 바뀐 부분만 return하면 나머지는 알아서 넘어감
    return {'is_ok': result}  


def normal_node(state: SystemState):
    print('----문제 없음 노드----', state)
    return {'next_step': '문제 없음. 계속 진행'}


def error_node(state: SystemState):
    print('----에러 발생 노드----', state)
    return {'next_step': '에러 발생. 삐용삐용'}


def call_admin(state: SystemState):
    print('---관리자 호출 노드---', state)
    return {'next_step': '관리자 호출!!!'}

In [ ]:
# Router (Node 아님 -> State 건들지 않기. 다음 Node를 판단할 값만 return)
def is_ok_router(state: SystemState):
    if state['is_ok'] == 'admin':
        return '관리자'
    elif state['is_ok'] == True:
        return '괜춘'
    else:
        return '안괜춘'

In [ ]:
builder = StateGraph(SystemState)

# 노드 등록
builder.add_node('메세지처리', process_message)
builder.add_node('일반노드', normal_node)
builder.add_node('에러노드', error_node)
builder.add_node('관리자노드', call_admin)

# 노드간 연결
builder.add_edge(START, '메세지처리')

# 라우터에 따라서 결정
builder.add_conditional_edges(
    '메세지처리',   # 이거 다음에 분기가 시작됨
    is_ok_router,  # 이 함수를 실행하고, 그 결과
    {
        '괜춘': '일반노드',  # 결과가 '괜춘'이면 '일반노드' 실행
        '안괜춘': '에러노드', # 결과가 '안괜춘'이면 '에러노드' 실행
        '관리자': '관리자노드',
    }
)
builder.add_edge('일반노드', END)
builder.add_edge('에러노드', END)
builder.add_edge('관리자노드', END)


graph = builder.compile()
graph


In [ ]:
# graph.invoke({'message': '좋아좋아'})
# graph.invoke({'message': 'Error!! - 비상비상'})
graph.invoke({'message': 'admin 은 응답하라'})